This notebook is the **domain reference** for the entire course. Every subsequent notebook uses the Fintech Bank tables as its data source — read this once to understand the business context, entity relationships, and where each table lives on disk.

## Business Overview

**Fintech Bank** is a mid-size digital bank operating in India with four business verticals:

| Vertical | Description |
|---|---|
| **Cards** | Credit and debit card issuance and transaction processing |
| **Loans** | Personal, home, and auto loan origination and repayment tracking |
| **Accounts** | Savings and current account management |
| **Payments** | UPI, NEFT, RTGS, and IMPS payment rails |

All four verticals share a single **customers** dimension — one customer can hold cards, loans, accounts, and make payments.

## Entity Relationships

```
customers (1)
    ├── card_accounts     (N)  ──▶  card_transactions    (N)
    ├── loan_accounts     (N)  ──▶  loan_repayments      (N)
    ├── bank_accounts     (N)  ──▶  account_transactions (N)
    └── payments          (N)
```

`customer_id` is the shared foreign key across all eight tables.

## Data Format Assignments

Each table is stored in a different format so the course covers every major Spark data source:

| Table | Format | Path | Rationale |
|---|---|---|---|
| `customers` | CSV | `data/customers/` | CRM / onboarding nightly export |
| `card_accounts` | JSON | `data/card_accounts/` | Card issuance API event payloads |
| `card_transactions` | Parquet | `data/card_transactions/` | High-volume columnar data lake landing |
| `loan_accounts` | JSON (multiline) | `data/loan_accounts/` | Loan origination system JSON documents |
| `loan_repayments` | CSV | `data/loan_repayments/` | Loan servicing platform flat-file export |
| `bank_accounts` | ORC | `data/bank_accounts/` | Legacy core banking / Hadoop pipeline output |
| `account_transactions` | ORC | `data/account_transactions/` | Core banking transaction ledger |
| `payments` | Delta | `data/payments/` | Real-time ACID payment processing |

> **Setup:** Run `generate_bank_data.ipynb` once to populate the `data/` folder before working through any other notebook.

## Schema Definitions

These are the canonical schemas. All notebooks import or re-declare them as needed.

### Design decisions

- Monetary values use `DecimalType(18, 2)` — no floating-point rounding errors
- Timestamps use `TimestampType`; calendar dates use `DateType`
- ~10 % of optional fields are intentionally `null` (real-world data is never clean)
- ~5 % of card / payment transactions are flagged `is_fraud = True`
- ~8 % of loan repayments have `status = MISSED`

In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, BooleanType,
    DecimalType, DateType, TimestampType, DoubleType,
)

# ── customers ──────────────────────────────────────────────────────────────
customers_schema = StructType([
    StructField("customer_id",   StringType(),        nullable=False),
    StructField("full_name",     StringType(),        nullable=False),
    StructField("email",         StringType(),        nullable=True),
    StructField("phone",         StringType(),        nullable=True),
    StructField("city",          StringType(),        nullable=True),
    StructField("state",         StringType(),        nullable=True),
    StructField("date_of_birth", DateType(),          nullable=True),
    StructField("kyc_verified",  BooleanType(),       nullable=False),
    StructField("created_at",    TimestampType(),     nullable=False),
])

# ── card_accounts ──────────────────────────────────────────────────────────
card_accounts_schema = StructType([
    StructField("card_id",       StringType(),        nullable=False),
    StructField("customer_id",   StringType(),        nullable=False),
    StructField("card_type",     StringType(),        nullable=False),  # CREDIT / DEBIT
    StructField("network",       StringType(),        nullable=True),   # VISA / MASTERCARD / RUPAY
    StructField("credit_limit",  DecimalType(18, 2),  nullable=True),
    StructField("issued_on",     DateType(),          nullable=False),
    StructField("expiry_date",   DateType(),          nullable=False),
    StructField("status",        StringType(),        nullable=False),  # ACTIVE / BLOCKED / EXPIRED
])

# ── card_transactions ──────────────────────────────────────────────────────
card_transactions_schema = StructType([
    StructField("txn_id",        StringType(),        nullable=False),
    StructField("card_id",       StringType(),        nullable=False),
    StructField("customer_id",   StringType(),        nullable=False),
    StructField("amount",        DecimalType(18, 2),  nullable=False),
    StructField("merchant",      StringType(),        nullable=True),
    StructField("category",      StringType(),        nullable=True),
    StructField("txn_ts",        TimestampType(),     nullable=False),
    StructField("status",        StringType(),        nullable=False),  # SUCCESS / DECLINED / REVERSED
    StructField("is_fraud",      BooleanType(),       nullable=False),
])

# ── loan_accounts ──────────────────────────────────────────────────────────
loan_accounts_schema = StructType([
    StructField("loan_id",       StringType(),        nullable=False),
    StructField("customer_id",   StringType(),        nullable=False),
    StructField("loan_type",     StringType(),        nullable=False),  # PERSONAL / HOME / AUTO
    StructField("principal",     DecimalType(18, 2),  nullable=False),
    StructField("interest_rate", DoubleType(),        nullable=False),  # annual %
    StructField("tenure_months", IntegerType(),       nullable=False),
    StructField("disbursed_on",  DateType(),          nullable=False),
    StructField("status",        StringType(),        nullable=False),  # ACTIVE / CLOSED / NPA
])

# ── loan_repayments ────────────────────────────────────────────────────────
loan_repayments_schema = StructType([
    StructField("repayment_id",  StringType(),        nullable=False),
    StructField("loan_id",       StringType(),        nullable=False),
    StructField("customer_id",   StringType(),        nullable=False),
    StructField("due_date",      DateType(),          nullable=False),
    StructField("paid_date",     DateType(),          nullable=True),   # null if missed
    StructField("emi_amount",    DecimalType(18, 2),  nullable=False),
    StructField("paid_amount",   DecimalType(18, 2),  nullable=True),
    StructField("status",        StringType(),        nullable=False),  # PAID / MISSED / PARTIAL
])

# ── bank_accounts ──────────────────────────────────────────────────────────
bank_accounts_schema = StructType([
    StructField("account_id",    StringType(),        nullable=False),
    StructField("customer_id",   StringType(),        nullable=False),
    StructField("account_type",  StringType(),        nullable=False),  # SAVINGS / CURRENT
    StructField("balance",       DecimalType(18, 2),  nullable=False),
    StructField("branch_code",   StringType(),        nullable=True),
    StructField("ifsc",          StringType(),        nullable=True),
    StructField("opened_on",     DateType(),          nullable=False),
    StructField("is_active",     BooleanType(),       nullable=False),
])

# ── account_transactions ───────────────────────────────────────────────────
account_transactions_schema = StructType([
    StructField("txn_id",        StringType(),        nullable=False),
    StructField("account_id",    StringType(),        nullable=False),
    StructField("customer_id",   StringType(),        nullable=False),
    StructField("txn_type",      StringType(),        nullable=False),  # CREDIT / DEBIT
    StructField("amount",        DecimalType(18, 2),  nullable=False),
    StructField("balance_after", DecimalType(18, 2),  nullable=False),
    StructField("description",   StringType(),        nullable=True),
    StructField("txn_ts",        TimestampType(),     nullable=False),
])

# ── payments ───────────────────────────────────────────────────────────────
payments_schema = StructType([
    StructField("payment_id",    StringType(),        nullable=False),
    StructField("customer_id",   StringType(),        nullable=False),
    StructField("payment_mode",  StringType(),        nullable=False),  # UPI / NEFT / RTGS / IMPS
    StructField("amount",        DecimalType(18, 2),  nullable=False),
    StructField("sender_vpa",    StringType(),        nullable=True),   # UPI VPA if applicable
    StructField("receiver_vpa",  StringType(),        nullable=True),
    StructField("status",        StringType(),        nullable=False),  # SUCCESS / FAILED / PENDING
    StructField("is_fraud",      BooleanType(),       nullable=False),
    StructField("initiated_at",  TimestampType(),     nullable=False),
    StructField("settled_at",    TimestampType(),     nullable=True),
])

print("All schemas defined — 8 tables across 5 formats.")

## Course Feature Roadmap

| Notebook | Domain usage |
|---|---|
| 07 — Reading & Writing Data | Read all 8 tables from their native formats (CSV, JSON, Parquet, ORC, Delta) |
| 08 — Core Transformations | Filter fraud transactions; cast `amount`; rename and drop columns |
| 09 — Aggregations & Window Functions | Revenue by vertical; running balance; payment rank per customer |
| 10 — Spark SQL & Temporary Views | SQL queries over all 8 tables via temp views |
| 11 — SQL Functions & UDFs | Mask PAN numbers; classify transaction amount buckets |
| 12 — Partitioning, Shuffles & Catalyst | Partition `card_transactions` by month; inspect shuffle plan |
| 13 — Caching & Persistence | Cache `customers` dimension for repeated joins |
| 14 — Broadcast Joins & Data Skew | Broadcast `card_accounts`; handle skew on `customer_id` |
| 15 — Structured Streaming Fundamentals | Simulate real-time payment event stream |
| 16 — Sources, Sinks & Watermarking | Watermark late-arriving payment events |
| 17 — Stateful Stream Processing | Session windows over card transaction stream |
| 18 — What is Delta Lake? | Introduce Delta on the `payments` table |
| 19 — ACID & Time Travel | Insert, update, delete payments; query previous versions |
| 20 — Delta Operations & Optimization | `OPTIMIZE`, `ZORDER`, `VACUUM` on `card_transactions` Delta |
| 21 — MLlib & Spark ML Pipelines | Fraud detection model on `card_transactions` |
| 22 — Databricks Certified Spark Developer Exam Guide | Full domain walkthrough as exam prep |